## Basic trying things (skip)

In [ ]:
import kdsource.api as kds
import numpy as np

In [ ]:
help(kds.write_mcpl)

In [ ]:
file_path = "my_data.mcpl.gz"

empty_10 = np.zeros(10)
full_10 = np.ones(10)
kds.write_mcpl(filename=file_path, ekin=full_10, x=empty_10, y=empty_10, z=empty_10, ux=full_10, uy=empty_10, uz=empty_10, time=empty_10, weight=full_10, pdgcode=2112)

In [ ]:
import mcpl
import pandas as pd

myfile=mcpl.MCPLFile(file_path,blocklength=2)
datas = []
for i,p in enumerate(myfile.particle_blocks):
    print("block no.", i)
    data = pd.DataFrame({
        'Kf': p.pdgcode,  # Particle PDG code
        'x': p.x,  # x-position
        'y': p.y,  # y-position
        'z': p.z,  # z-position
        'Vx': p.ux,  # Velocity in x-direction
        'Vy': p.uy,  # Velocity in y-direction
        'Vz': p.uz,  # Velocity in z-direction
        'e': p.ekin,  # Kinetic energy
        'Wt': p.weight  # Weight of the particle
    })
    datas.append(data)
data = pd.concat(datas, ignore_index=True)
data

In [ ]:
data.dtypes

In [ ]:
myfile=mcpl.MCPLFile("./examples/uniform/output.mcpl.gz",blocklength=100)
datas = []
for i,p in enumerate(myfile.particle_blocks):
    if i==1: break
    data = pd.DataFrame({
        'Kf': p.pdgcode,  # Particle PDG code
        'x': p.x,  # x-position
        'y': p.y,  # y-position
        'z': p.z,  # z-position
        'Vx': p.ux,  # Velocity in x-direction
        'Vy': p.uy,  # Velocity in y-direction
        'Vz': p.uz,  # Velocity in z-direction
        'e': p.ekin,  # Kinetic energy
        'Wt': p.weight  # Weight of the particle
    })
    datas.append(data)
data = pd.concat(datas, ignore_index=True)
data

In [ ]:
kds.write_mcpl(filename=file_path, ekin=data["e"], x=data["x"], y=data["y"], z=data["z"],
                ux=data["Vx"], uy=data["Vy"], uz=data["Vz"], time=np.zeros_like(data["Kf"]), weight=data["Wt"], pdgcode=data["Kf"])

In [ ]:
import kdsource.api as kds  # Importing the KDSource Python API

In [ ]:
kds.Geometry?

In [ ]:
kds.Metric?

In [ ]:
kds.geom._metrics

In [ ]:
my_energy = kds.geom.Energy()
#my_lethargy = kds.geom.Lethargy()

In [ ]:
type(my_energy)

In [ ]:
for k,v in my_energy.__dict__.items():
    print(k, str(v))

In [ ]:
# for k,v in my_lethargy.__dict__.items():
#     print(k, str(v))

In [ ]:
my_position = kds.geom.Vol()
my_time = kds.geom.Time()
my_directions = kds.geom.Isotrop()

In [ ]:
my_geometry = kds.Geometry([
    my_energy, my_time, my_position, my_directions
])

In [ ]:
for k,v in my_geometry.__dict__.items():
    print(k, str(v))

In [ ]:
kds.KDSource?

In [ ]:
import kdsource

In [ ]:
kdsource?

In [ ]:
kds.kde.bw_methods

In [ ]:
# Step 1: Load your original MCPL particle track list file
my_input_file_path = "./examples/uniform/output.mcpl.gz"

my_input_plist = kds.PList(filename=my_input_file_path,
                           readformat="mcpl",
                           pt="n",
                           trasl=None,
                           rot=None,
                           switch_x2z=False,
                           set_params=True)


In [ ]:
vars(my_input_plist)

In [ ]:
my_source.fitted

In [ ]:
my_source = kds.KDSource(
    plist=my_input_plist, geom=my_geometry, bw="mlcv", J=1.0, kernel="gaussian"
)

In [ ]:
my_source.fitted

In [ ]:
[attr for attr in dir(my_source) if attr[0:2]!="__"]

In [ ]:
vars(my_source.plist)

In [ ]:
my_source.fitted, my_source.bw_method

In [ ]:
my_source.fit(N=10000)

In [ ]:
vars(my_source.geom)

In [ ]:
my_source.kde.fit?

In [ ]:
my_source.fitted

In [ ]:
import numpy as np
my_source.plot_t(np.array([-1,0,1]))

In [ ]:
my_source.kde.data

In [ ]:
my_source.plot_E(grid_E=np.array([0,1e-8,2e-8,3e-8]))

In [ ]:
my_source.plot_integr(var="x", grid=np.arange(-5,5,0.2));

In [ ]:
my_source.save()

In [ ]:
kds.resample_to_mcpl("_source.xml", "generated_from_source.mcpl.gz", nout=100)

In [ ]:
import mcpl
import pandas as pd

gen_file=mcpl.MCPLFile("generated_from_source.mcpl.gz")
datas = []
for i,p in enumerate(gen_file.particle_blocks):
    print("block no.", i)
    data = pd.DataFrame({
        'Kf': p.pdgcode,  # Particle PDG code
        'x': p.x,  # x-position
        'y': p.y,  # y-position
        'z': p.z,  # z-position
        'Vx': p.ux,  # Velocity in x-direction
        'Vy': p.uy,  # Velocity in y-direction
        'Vz': p.uz,  # Velocity in z-direction
        'e': p.ekin,  # Kinetic energy
        'Wt': p.weight  # Weight of the particle
    })
    datas.append(data)
data = pd.concat(datas, ignore_index=True)
data

In [ ]:
my_input_plist.N

In [ ]:
kdsource.plist.pt2pdg?

# methods (skip)
read_data, resample_df, resample_df_gen, generate_data, plot_data_row

## method definitions

In [5]:
import kdsource.api as kds
import mcpl
import pandas as pd
from pathlib import Path
import tempfile
import numpy as np


def is_constant_column(column):
    return np.all(column == column[0], axis=0)


def read_data(file_path, n=-1):
    """
    Read up to n particles from the mcpl file. n==-1 means read all.
    """
    blocklength = 10000
    gen_file = mcpl.MCPLFile(file_path, blocklength=blocklength)
    datas = []
    for i, p in enumerate(gen_file.particle_blocks):
        data = pd.DataFrame(
            {
                "ekin": p.ekin,  # Kinetic energy
                "x": p.x,  # x-position
                "y": p.y,  # y-position
                "z": p.z,  # z-position
                "ux": p.ux,  # Velocity in x-direction
                "uy": p.uy,  # Velocity in y-direction
                "uz": p.uz,  # Velocity in z-direction
                "time": p.time,
                "weight": p.weight,  # Weight of the particle
                "pdgcode": p.pdgcode,  # Particle PDG code
            }
        )
        datas.append(data)
        if n != -1 and (i + 1) * blocklength >= n:
            break
    data = pd.concat(datas, ignore_index=True)
    if n != -1 and len(data) > n:
        data = data[:n]
    return data

def resample_df(
    source,
    dest,
    nout,
    rng=None,
    force=False,
    no_df=False,
    del_dest=False,
    del_source=False,
):
    """
    Generates data at `dest` and by default returns it as a pandas dataframe.
    - source: source model .xml file (which contains the original data path that is required for sampling)
    - dest: a file path with .mcpl.gz ending
    - nout: amount of particles generated
    - rng: rng parameter
    - force: whether `dest` file may be overwritten if it already exists
    - no_df: whether you do NOT want to receive a dataframe for some reason
    - del_dest: whether to delete file at `dest` in the end
    - del_source: whether or not to delete the source .xml file in the end

    Returns
    - pandas dataframe, or `dest`, or None
    """
    # print("Amount of particles to generate:", str(nout))
    kds.resample_to_mcpl(
        kds_sourcefile=source,
        destination_mcpl=dest,
        nout=nout,
        rng=rng,
        force=force,
    )
    # print("Data generated at", dest)
    if not no_df:
        data = read_data(dest)
    if del_dest:
        Path(dest).unlink()
    if del_source:
        Path(source).unlink()
    if not no_df:
        return data
    if not del_dest:
        return dest
    return None

def resample_df_gen(source, dest, nout, rng=None, del_dest=False, del_source=False):
    """
    A generator for dataframes via the resample method using an intermediate mcpl.gz file at `dest`. ATTENTION: Overwrites `dest` in each iteration!
    - source: source model .xml file (which contains the original data path that is required for sampling)
    - dest: a file path with .mcpl.gz ending
    - nout: amount of particles per dataframe
    - rng: If you specify it, it will be used for all the generated data. If None, uses new random rng for each iteration.
    - del_dest: whether or not to delete the most recent intermediate mcpl.gz file on the disk
    - del_source: whether or not to delete the source .xml file in the end (after the generator has stopped)
    """
    try:
        while True:
            yield resample_df(
                source=source,
                dest=dest,
                nout=nout,
                rng=rng,
                force=True,
                no_df=False,
                del_dest=del_dest,
                del_source=False,
            )
    finally:
        if del_source:
            Path(source).unlink()




def generate_data(
    inputFile,
    output_path=None,
    overwrite_output=False,
    n_gen=1_000,
    pt="n",
    n_fit=-1,
    kernel="gaussian",
    bw="silv",
    pCurrent=1.0,
    model_path=None,
    overwrite_model=False,
    generator=False,
):
    """
    Generates data similar to inputFile using kdsource, which uses KDE (Kernel Density Estimation) modeling.

    Parameters
    ---
    - inputFile: file path to input data; requires ending `.mcpl` or `.mcpl.gz`
    - output_path: file path for generated data (optional, see "Returns" section); file ending will be `.mcpl.gz`
    - overwrite_output: whether already existing file at output_path may be overwritten
    - n_gen: amount of particles to generate
    - pt: particle type ("n", "p", or "e") of generated particles and of input particles used for fitting the KDE model
    - n_fit: amount of input particles (with type `pt`) used for fitting of the source model (-1 means all)
    - kernel: kernel shape ("gaussian", "box", or "epa" (Epanechnikov))
    - bw: method for finding kernel bandwidth ("silv", "knn" or "mlcv")
    - pCurrent: total source particle current
    - model_path: where to save the model xml file (None makes it temporary)
    - overwrite_model: whether already existing file at model_path may be overwritten
    - generator: whether you want a generator for dataframes instead of one dataframe. `n_gen` will be the amount per yield.

    Returns
    ---
    - If output_path is specified: None. (The data will be saved to the file.)
    - If output_path is None: Pandas dataframe. Or, if `generator` is True: Generator of pandas dataframes.
    """

    inputFile = str(Path(inputFile).absolute())

    return_a_df = False
    output_is_temp = False
    model_is_temp = False

    if output_path is not None:
        output_path = Path(output_path).absolute().with_suffix(".mcpl.gz")
        if output_path.exists() and not overwrite_output:
            raise ValueError(
                "Output file already exists and overwriting is not allowed"
            )
        output_path = str(output_path)
    else:
        return_a_df = True
        output_path = str(Path(tempfile.gettempdir()) / "brightify_kdsource.mcpl.gz")
        output_is_temp = True

    if model_path is not None:
        model_path = Path(model_path).absolute().with_suffix(".xml")
        if model_path.exists() and not overwrite_model:
            raise ValueError("Model file already exists and overwriting is not allowed")
        model_path = str(model_path)
    else:
        model_path = str(Path(tempfile.gettempdir()) / "brightify_kdsource.xml")
        model_is_temp = True

    my_input_plist = kds.PList(
        filename=inputFile,
        readformat="mcpl",
        pt=pt,
        trasl=None,
        rot=None,
        switch_x2z=False,
        set_params=True,
    )
    # print(vars(my_input_plist))

    my_energy = kds.geom.Energy()
    my_time = kds.geom.Time()
    my_position = kds.geom.Vol()
    my_directions = kds.geom.Isotrop()
    my_geometry = kds.Geometry([my_energy, my_time, my_position, my_directions])
    # print(vars(my_geometry))

    my_source = kds.KDSource(
        plist=my_input_plist, geom=my_geometry, bw=bw, J=pCurrent, kernel=kernel
    )
    # print(vars(my_source))

    if n_fit == -1 or n_fit > my_input_plist.N:
        n_fit = my_input_plist.N
    # print("Amount of particles for fitting:", n_fit)

    my_source.fit(N=n_fit)

    model_path = str(Path(my_source.save(xmlfilename=model_path)).absolute())
    # if not model_is_temp:
    #     print("Source model saved to", model_path)
    # else:
    #     print("Temporary source model saved to", model_path)



    if not return_a_df:
        kds.resample_to_mcpl(
            kds_sourcefile=model_path,
            destination_mcpl=output_path,
            nout=n_gen,
            rng=None,
            force=True,
        )
        if model_is_temp:
            Path(model_path).unlink()
        return output_path
    else:
        if not generator:
            return resample_df(
                source=model_path,
                dest=output_path,
                nout=n_gen,
                rng=None,
                force=True,
                no_df=False,
                del_dest=output_is_temp,
                del_source=model_is_temp,
            )
        else:
            return resample_df_gen(
                source=model_path,
                dest=output_path,
                nout=n_gen,
                rng=None,
                del_dest=output_is_temp,
                del_source=model_is_temp,
            )

In [6]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def plot_data_row(datas, rows=["x", "ux", "ekin", "weight"]):
    colors = list(mcolors.TABLEAU_COLORS.values())
    for j, row in enumerate(rows):
        color = colors[j]
        for i, data in enumerate(datas):
            plt.hist(data[row], density=True, bins=20, histtype="step", label=row+" "+str(i), color=color, linestyle=(0,(i%2+1,i*2+1)))
    plt.legend()
    plt.show()

## examples (skip)

In [ ]:
example_path = "./examples/uniform/output.mcpl.gz"

In [ ]:
data1 = generate_data(inputFile=example_path, n_gen=100_000, n_fit=1_000_000, bw="silv")

In [ ]:
data2 = read_data(example_path, 100_000)

In [ ]:
gen = generate_data(
    inputFile=example_path,
    output_path=None,
    overwrite_output=False,
    n_gen=100,
    pt="n",
    n_fit=100_000,
    kernel="gaussian",
    bw="silv",
    pCurrent=1.0,
    model_path=None,
    overwrite_model=False,
    generator=True,
)

In [ ]:
data3=next(gen)

In [ ]:
gen.close()

In [ ]:
data3=next(gen)

In [ ]:
import numpy as np

In [ ]:
for d in [data1, data2, data3]:
    d["V_norm"] = np.sqrt((d.Vx*d.Vx + d.Vy*d.Vy + d.Vz+d.Vz))

In [ ]:
data1.describe()

In [ ]:
data2.describe()

In [ ]:
data3.describe()

In [ ]:
plot_data_row(datas=[data1, data2, data3], rows=["e", "x", "Vx"])

# Exercise 26.10. - 30.10.

## file paths and manual inspection (skip)

In [9]:
example_path_1 = "./examples/Brightify Fig 6/Fig6a/output.mcpl.gz"  # const: ekin=2.5e-08, weight=1; almost const: z=0.06; n = 20_139_128
example_path_2 = "./examples/Brightify Fig 6/Fig6b/output.mcpl.gz"  # const: ekin=2.5e-08, weight=1; almost const: z=0.06; n = 21_580_959 (SAME AS 5)
example_path_3 = "./examples/Brightify Fig 6/Fig6c/output.mcpl.gz"  # const: ekin=2.5e-08, weight=1; almost const: z=0.06; n = 11_002_813
example_path_4 = "./examples/mcpl_different_normal/dump_final.mcpl.gz"  # almost const: x=13.718; n = 2_738_810
example_path_5 = "./examples/uniform/output.mcpl.gz"  # const: ekin=2.5e-08, weight=1; almost const: z=0.06; n = 21_580_959 (SAME AS 2)
example_path_6 = "./examples/resampled.mcpl.gz"  # const: z=286.915, weight=1; n = 2_000_000

In [ ]:
# !pymcpltool --stats "$example_path_4"

In [ ]:
# !pymcpltool --stats --gui "$example_path_6" # it did not work inside the notebook here, but from terminal (because of WSL)

## calculate silverman & mlcv and gaussian & epa, for 2 different data (skip)

### declare files, plist parameters and geometries

In [10]:
example_paths = {
    1: example_path_1,
    2: example_path_2,
    3: example_path_3,
    4: example_path_4,
    5: example_path_5, # is the same as 2
    6: example_path_6,
}

# plist parameters:
pt = {index:"n" for index in example_paths.keys()}
trasl = {}
rot = {}
switch_x2z = {4: True} # swaps x to z, y to x, z to y

geometries = { # one per file
    1: kds.geom.Geometry([kds.geom.SurfXY(z=0.06)]), # omitting all dimensions except x and y position
    2: kds.geom.Geometry([kds.geom.SurfXY(z=0.06)]),
    3: kds.geom.Geometry([kds.geom.SurfXY(z=0.06)]),
    4: kds.geom.GeomFlat(z=13.718, E0=8.18e-08),
    5: kds.geom.Geometry([kds.geom.SurfXY(z=0.06)]),
    6: kds.geom.GeomFlat(z=286.915, E0=10),
}

### choose what to calculate

In [ ]:
bw_kernel_combinations = {  # choose from these by referencing the number/index below
    1: ("silv", "gaussian"),
    2: ("silv", "epa"),
    3: ("mlcv", "gaussian"),
    4: ("mlcv", "epa"),
}

calculate_these = [  # format: (file_index, bw_kernel_index, n_fit)
    # where n_fit is the number of particles (after "pt" filtering) to use for the fit.
    # type -1 for n_fit to use all the particles. use a value for n_fit between 0 and 1 to use a certain percentage
    # (5, 1, 10_000),
    # (5, 2, 10_000),
    # (5, 3, 10_000),
    # (5, 4, 10_000),
    # (6, 1, 10_000),
    # (6, 2, 10_000),
    # (6, 3, 10_000),
    # (6, 4, 10_000),
]
calculate_these.extend(
    [
        (fn, bw_kernel_index, n_fit)
        for fn in (5,6)
        for bw_kernel_index in (3,4)
        for n_fit in (
            500_000,
        )
    ]
)

# calculate_these.extend(
#     [
#         (fn, bw_kernel_index, n_fit)
#         for fn in (5,6)
#         for bw_kernel_index in (1,2)
#         for n_fit in (
#             1_000,
#             5_000,
#             10_000,
#             20_000,
#             50_000,
#             100_000,
#             200_000,
#             500_000,
#             1_000_000,
#             2_000_000,
#         )
#     ]
# )

calculate_these

### Create partial data files to use for fitting

In [ ]:
copy_data_folder = "./copied_data"

In [ ]:
from pathlib import Path
copy_data_folder = Path(copy_data_folder).absolute()
if not copy_data_folder.exists():
    Path.mkdir(copy_data_folder)

In [ ]:
import numpy as np

copy_files = {(int(fn), int(n_fit)): str(p) for fn, n_fit, p in [str(p.with_suffix("").stem).split("_")+[p] for p in copy_data_folder.glob("*.mcpl.gz")]}
print(copy_files)

for (fn, bw_kernel_num, n_fit) in calculate_these:
    file = example_paths[fn]
    copy_file = str(copy_data_folder / Path(str(fn)+"_"+str(n_fit)+".mcpl.gz"))
    fn_n_fit = (fn, n_fit)
    if fn_n_fit in copy_files.keys():
        continue
    copy_data = read_data(file_path=file, n=n_fit)
    kds.write_mcpl(copy_file,
                    ekin=copy_data["ekin"],
                    x=copy_data["x"],
                    y=copy_data["y"],
                    z=copy_data["z"],
                    ux=copy_data["ux"],
                    uy=copy_data["uy"],
                    uz=copy_data["uz"],
                    time=copy_data["time"],
                    weight=copy_data["weight"][0] if is_constant_column(copy_data["weight"]) else copy_data["weight"],
                    pdgcode=copy_data["pdgcode"][0] if is_constant_column(copy_data["pdgcode"]) else copy_data["pdgcode"],
                    polx=None,
                    poly=None,
                    polz=None,
                    userflags=None,
                    )
    copy_files[fn_n_fit] = copy_file


### run calculations, to obtain model xml files

In [ ]:
models_folder="./calculated_models"

In [ ]:
from pathlib import Path
models_folder = Path(models_folder).absolute()
if not models_folder.exists():
    Path.mkdir(models_folder)

In [ ]:
import math
import time
import numpy as np

for i, (fn, bw_kernel_num, n_fit) in enumerate(calculate_these):
    
    fn_n_fit = (fn, n_fit)
    plist = kds.PList(
        filename=copy_files[fn_n_fit],
        readformat="mcpl",
        pt=pt[fn] if fn in pt.keys() else "n",
        trasl=trasl[fn] if fn in trasl.keys() else None,
        rot=rot[fn] if fn in rot.keys() else None,
        switch_x2z=switch_x2z[fn] if fn in switch_x2z.keys() else False,
        set_params=True)

    N = plist.N

    bw, kernel = bw_kernel_combinations[bw_kernel_num]
    if n_fit == -1: n_fit = N
    elif 0 < n_fit < 1: n_fit = math.floor(N * n_fit)
    else: n_fit = max(min(N, n_fit), 0)
    if n_fit == 0:
        print("n_fit became zero, calculation will be skipped now.")
        continue

    source = kds.KDSource(
            plist=plist,
            geom=geometries[fn],
            bw=bw,
            kernel=kernel)
    
    fit_success = False
    start = time.perf_counter()
    mlcv_grid_multiplier = 5
    mlcv_grid_start = (-0.1 if n_fit < 100_000 else -0.5) if n_fit < 500_000 else -2.5
    mlcv_grid_stop = (0.1 if n_fit < 100_000 else 0.5) if n_fit < 500_000 else 2.5
    mlcv_grid_num = 8
    mlcv_attempt = 1    
    while fit_success == False and mlcv_attempt <= 3:
        try:
            if bw=="mlcv":
                grid = np.logspace(mlcv_grid_start, mlcv_grid_stop, mlcv_grid_num)
                print("attempting grid logspace start stop", str(mlcv_grid_start), str(mlcv_grid_stop))
                source.fit(N=n_fit, grid=grid)
            else:
                source.fit(N=n_fit)
        except:
            print("Fitting failed")
            if bw=="mlcv":
                mlcv_grid_start *= mlcv_grid_multiplier
                mlcv_grid_stop *= mlcv_grid_multiplier
        else:
            fit_success = True
        finally:
            if bw=="mlcv":
                mlcv_attempt += 1
            else:
                break        
    end = time.perf_counter()
    elapsed_seconds = int(end-start)
    print(f"Elapsed: {elapsed_seconds} s")

    if fit_success:
        xmlfilename = str(models_folder)+"/file_"+str(fn)+"_"+bw+"_"+kernel+"_"+str(N)+"_"+str(n_fit)+"_"+str(elapsed_seconds)+"s.xml"
        source.save(xmlfilename=xmlfilename)


### generate (resample) data from the models

In [2]:
gen_N = 2_000_000 # number of particles to generate per model
gen_data_folder = "./generated_data"
RNG = 123456789

In [3]:
from pathlib import Path

gen_data_folder = Path(gen_data_folder).absolute()
if not gen_data_folder.exists():
    Path.mkdir(gen_data_folder)

In [ ]:
for model_path in models_folder.glob("*"):
    #print(model_path)
    stem = model_path.with_suffix("").stem
    f, fn, bw, kernel, N, n_fit, elapsed = stem.split(sep="_")
    # print(f, fn, bw, kernel, N, n_fit, elapsed)
    fn = int(fn)
    N = int(N)
    n_fit = int(n_fit)
    dest_path = gen_data_folder / Path(stem).with_suffix(".mcpl.gz")
    # print(dest_path)
    if dest_path in gen_data_folder.glob("*"):
        print("skipping", dest_path)
        continue
    resample_df(source=model_path, dest=dest_path, nout=gen_N, rng=RNG, force=True, no_df=True, del_dest=False, del_source=False)

### read data and produce metrics

In [ ]:
from collections import defaultdict
def nested_dict():
    return defaultdict(nested_dict)
def normalize_nested_dict(d):
    if isinstance(d, defaultdict):
        return {k: normalize_nested_dict(v) for k, v in d.items()}
    return d

gen_data_metrics = nested_dict()
for gen_data_path in gen_data_folder.glob("*"):
    stem = gen_data_path.with_suffix("").stem
    f, fn, bw, kernel, N, n_fit, elapsed = stem.split(sep="_")
    # print(elapsed)
    fn = int(fn)
    bw_kernel = (bw, kernel)
    n_fit = int(n_fit)
    elapsed = int(elapsed.rstrip("s"))
    gen_data_metrics[fn][bw_kernel][n_fit]['elapsed'] = elapsed
    gen_data = read_data(gen_data_path)
    summary = gen_data.describe().loc[['mean', 'std', 'min', 'max']]
    for c in summary.columns:
        gen_data_metrics[fn][bw_kernel][n_fit]['mean'][c] = summary.loc['mean', c]
        gen_data_metrics[fn][bw_kernel][n_fit]['std'][c] = summary.loc['std', c]
        gen_data_metrics[fn][bw_kernel][n_fit]['min'][c] = summary.loc['min', c]
        gen_data_metrics[fn][bw_kernel][n_fit]['max'][c] = summary.loc['max', c]
gen_data_metrics = normalize_nested_dict(gen_data_metrics)

original_data_metrics = nested_dict()
for fn in gen_data_metrics.keys():
    original_path = example_paths[fn]
    data = read_data(original_path)
    summary = data.describe().loc[['mean', 'std', 'min', 'max']]
    for c in summary.columns:
        original_data_metrics[fn]['mean'][c] = summary.loc['mean', c]
        original_data_metrics[fn]['std'][c] = summary.loc['std', c]
        original_data_metrics[fn]['min'][c] = summary.loc['min', c]
        original_data_metrics[fn]['max'][c] = summary.loc['max', c]
original_data_metrics = normalize_nested_dict(original_data_metrics)

NameError: name 'example_paths' is not defined

In [22]:
bin_count = 100
bin_extra = 10 # amount of extra bins left and right of min and max found in original data (to account for generated data values outside of it)

bins_min = nested_dict()
bins_max = nested_dict()
for fn, gdm_fn in gen_data_metrics.items():
    for c, v in original_data_metrics[fn]['max'].items():
        if not c in bins_max[fn] or bins_max[fn][c] < v:
            bins_max[fn][c] = v
    for c, v in original_data_metrics[fn]['min'].items():
        if not c in bins_min[fn] or bins_min[fn][c] > v:
            bins_min[fn][c] = v

bins = nested_dict()
orig_hists = nested_dict()
for fn in bins_min.keys():
    for c in bins_min[fn].keys():
        ma = bins_max[fn][c]
        mi = bins_min[fn][c]
        span = ma - mi
        per_bin = span/bin_count
        extra_span = per_bin * bin_extra
        ma += extra_span
        mi -= extra_span
        bins[fn][c] = np.linspace(mi, ma, bin_count + 2*bin_extra + 1)
    original_path = example_paths[fn]
    data = read_data(original_path)
    for c, b in bins[fn].items():
        orig_hists[fn][c] = np.histogram(data[c], bins=b)
bins = normalize_nested_dict(bins)
orig_hists = normalize_nested_dict(orig_hists)

gen_hists = nested_dict()
for gen_data_path in gen_data_folder.glob("*"):
    stem = gen_data_path.with_suffix("").stem
    f, fn, bw, kernel, N, n_fit, elapsed = stem.split(sep="_")
    # print(elapsed)
    fn = int(fn)
    bw_kernel = (bw, kernel)
    n_fit = int(n_fit)
    gen_data = read_data(gen_data_path)
    for c in bins[fn].keys():
        gen_hists[fn][bw_kernel][n_fit][c] = np.histogram(gen_data[c], bins[fn][c])
gen_hists = normalize_nested_dict(gen_hists)

In [27]:
gen_hists

{5: {('silv',
   'epa'): {100000: {'ekin': (array([      0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,     

## save / load

In [1]:
results_dir = "./pickled_results"

In [2]:
import pickle as pkl
from pathlib import Path
results_dir = Path(results_dir).absolute()
    
gen_data_file = str(results_dir / "gen_data.pkl")
orig_data_file = str(results_dir / "orig_data.pkl")
gen_hists_file = str(results_dir / "gen_hists.pkl")
orig_hists_file = str(results_dir / "orig_hists.pkl")
print(gen_data_file, orig_data_file, gen_hists_file, orig_hists_file)

/home/jhonal/Brightify/pickled_results/gen_data.pkl /home/jhonal/Brightify/pickled_results/orig_data.pkl /home/jhonal/Brightify/pickled_results/gen_hists.pkl /home/jhonal/Brightify/pickled_results/orig_hists.pkl


#### save pickle (skip)

In [31]:
if not results_dir.exists():
    Path.mkdir(results_dir)
with open(gen_data_file, "wb") as f:
    pkl.dump(gen_data_metrics, f)
with open(orig_data_file, "wb") as f:
    pkl.dump(original_data_metrics, f)
with open(gen_hists_file, "wb") as f:
    pkl.dump(gen_hists, f)
with open(orig_hists_file, "wb") as f:
    pkl.dump(orig_hists, f)

### load pickle

In [3]:
with open(gen_data_file, "rb") as f:
    gen_data_metrics = pkl.load(f)
with open(orig_data_file, "rb") as f:
    original_data_metrics = pkl.load(f)
with open(gen_hists_file, "rb") as f:
    gen_hists = pkl.load(f)
with open(orig_hists_file, "rb") as f:
    orig_hists = pkl.load(f)

In [ ]:
# gen_data_metrics

In [ ]:
# original_data_metrics

In [33]:
gen_hists

{5: {('silv',
   'epa'): {100000: {'ekin': (array([      0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,       0,       0,
                  0,       0,       0,       0,       0,     

In [34]:
orig_hists

{5: {'ekin': (array([       0,        0,        0,        0,        0,        0,
                 0,        0,        0,        0,        0,        0,
                 0,        0,        0,        0,        0,        0,
                 0,        0,        0,        0,        0,        0,
                 0,        0,        0,        0,        0,        0,
                 0,        0,        0,        0,        0,        0,
                 0,        0,        0,        0,        0,        0,
                 0,        0,        0,        0,        0,        0,
                 0,        0,        0,        0,        0,        0,
                 0,        0,        0,        0,        0,        0,
                 0,        0,        0,        0,        0,        0,
                 0,        0,        0,        0,        0,        0,
                 0,        0,        0,        0,        0,        0,
                 0,        0,        0,        0,        0,        0,
         

## Visualize metrics

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

file_widget = widgets.Dropdown(options=list(gen_data_metrics.keys()), description='file no.:')
prop_widget = widgets.Dropdown(options=["mean", "std", "min", "max", "hist", "elapsed"], description='property:')
dim_widget = widgets.Dropdown(options=["ekin", "x", "y", "z", "ux", "uy", "uz", "weight", "pdgcode"], description='dimension:')

plot_button = widgets.Button(description="Generate Plot", button_style='primary')

out = widgets.Output()

_is_running = False

def on_button_clicked(b):
    global _is_running
    if _is_running:
        return

    _is_running = True
    b.description = "Plotting..."
    b.disabled = True

    try:
        with out:
            clear_output(wait=True)

            file = file_widget.value
            prop = prop_widget.value
            dimension = dim_widget.value

            if prop == "hist":
                gen_h = gen_hists[file]
                orig_h = orig_hists[file]
                fig3, ax3 = plt.subplots(figsize=(6*2, 4*2), constrained_layout=True)
                for (bw, kernel), gen_h_bw in gen_h.items():
                    for n_fit, gen_h_n in gen_h_bw.items():
                        counts, edges = gen_h_n[dimension]
                        norm_counts = counts/np.sum(counts)
                        ax3.stairs(norm_counts, edges, label="gen. data "+bw+" "+kernel+" , n_fit: "+str(n_fit)) 
                counts, edges = orig_h[dimension]
                norm_counts = counts/np.sum(counts)
                ax3.stairs(norm_counts, edges, label="orig. data")            
                ax3.set_xlabel(dimension+" histogram")
                ax3.set_ylabel("counts (normalized)")
                ax3.legend()
                ax3.grid()
                plt.show()
                plt.close(fig3)
            else:
                gen_metrics = gen_data_metrics[file]
                orig_metrics = original_data_metrics[file]
                bw_kernels = list(gen_metrics.keys())
                bws = sorted(list({bw_kernel[0] for bw_kernel in bw_kernels}))
                kernels = sorted(list({bw_kernel[1] for bw_kernel in bw_kernels}))
                # print(bws, kernels)
                n_bws, n_kernels = len(bws), len(kernels)
                # print(n_bws, n_kernels)
                fig, axs = plt.subplots(n_bws, n_kernels, figsize=(6*n_kernels, 4*n_bws), squeeze=False, constrained_layout=True)
                fig2, ax2 = plt.subplots(figsize=(6*n_kernels, 4*n_bws), constrained_layout=True)              
                ylabel=""
                for i_bw, bw in enumerate(bws):
                    for i_kernel, kernel in enumerate(kernels):
                        ax = axs[i_bw, i_kernel]
                        bw_kernel = (bw, kernel)
                        metrics = gen_metrics[bw_kernel]
                        n_fits = sorted(list(metrics.keys()))
                        # print(n_fits)
                        ylabel = prop
                        if prop == "elapsed":
                            prop_values = [metrics[n_fit][prop] for n_fit in n_fits]
                            # print(prop_values)
                            ylabel += " (s)"
                            ax.set_yscale("symlog")
                            if i_bw == 0 and i_kernel == 0: ax2.set_yscale("symlog")
                        else:
                            prop_values = [metrics[n_fit][prop][dimension] for n_fit in n_fits]
                            ylabel += " "+dimension
                            orig_value = orig_metrics[prop][dimension]
                            ax.axhline(orig_value, color='gray', linestyle='--', label="orig. data")
                            if i_bw == 0 and i_kernel == 0: ax2.axhline(orig_value, color='gray', linestyle='--', label="orig. data")
                        ax.plot(n_fits, prop_values, marker='o', linestyle="-.", label="gen. data")
                        ax2.plot(n_fits, prop_values, marker='o', linestyle="-.", label="gen. data "+bw+" "+kernel)
                        ax.set_title(bw+" "+kernel)
                        ax.set_xlabel("n_fit")
                        ax.set_ylabel(ylabel)
                        ax.set_xscale("log")
                        ax.legend()
                        ax.grid()
                        ax.set_xscale("log")
                fig.suptitle("file no. "+str(file))
                ax2.set_xlabel("n_fit")
                ax2.set_ylabel(ylabel)
                ax2.legend()
                ax2.grid()          
                plt.show()
                plt.close(fig)
                plt.close(fig2)
            return
    finally:
        # Always release the lock and restore the button even if something errors
        _is_running = False
        b.description = "Generate Plot"
        b.disabled = False

# Link the button click to the function
plot_button._click_handlers.callbacks.clear()
plot_button.on_click(on_button_clicked)

# Display the UI block
display(widgets.VBox([file_widget, prop_widget, dim_widget, plot_button, out]))

### Observations

1. The silverman method is incredibly fast compared to mlcv, whose run time scales approximately as n_fit^2 (quadratic with the amount of particles used for fitting).
2. Regarding the mean values, silverman vs. mlcv difference can not be seen very well. For file 5, at n_fit =~ 10^4, the mean of x and y meet the original data mean for the first time and then touch / cross it, getting sort of closer to it with increasing n_fit. While for file 6, x and y they come close to the true average at around n_fit = 10^5, and before that cross through it, showing that mere 1000 particles are definitely insufficient. All that independently of whether silverman of mlcv is looked at.
3. Between gaussian and epa kernel, no large difference is observable, except that the variance/std of the values is a bit smaller for the Epanechnikov kernel.
